# ANN — Student Exam Score Prediction (A to Z, with Activation Function Comparison)

**Goal:** Student ke study habits (hours studied, attendance, sleep, previous score, practice tests, extracurricular hours) se **final exam score predict karna** — ye ek **regression problem** hai (output ek continuous number hai, category nahi).

Is notebook mein:
1. Data load aur explore karenge
2. Data ko train/test mein split aur scale karenge
3. Same ANN architecture pe **alag-alag activation functions** (`relu`, `sigmoid`, `tanh`, `linear`) ek-ek karke try karenge
4. Dekhenge kaunsa activation sahi predict kar raha hai aur kaunsa fail (flat/constant output) ho raha hai
5. Sabka result compare karke best activation choose karenge

## Step 1 — Libraries import karo

- `pandas`, `numpy` → data handle karne ke liye
- `keras`/`tensorflow` → ANN banane ke liye
- `sklearn` ke functions → data split, scaling, aur accuracy check karne ke liye

In [1]:
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras.models import Sequential
from keras.layers import Dense, Activation

## Step 2 — Dataset load karo

`student_exam_score.csv` ko ek pandas DataFrame (table jaisi structure) mein load kar rahe hain.

In [2]:
dataset = pd.read_csv("student_exam_score.csv")
dataset.head()

,hours_studied,attendance_pct,sleep_hours,previous_score,practice_tests,extracurricular_hours,final_score
0,7.50,99.83,7.80,39.96,10,3.91,88.43
1,10.77,49.40,6.56,45.41,12,3.03,87.82
2,9.31,88.77,4.81,68.75,2,7.67,91.48
3,2.70,64.67,5.08,90.88,9,8.42,74.64
4,3.60,87.00,5.90,60.67,3,8.77,71.90


## Step 3 — Data explore karo

- `.shape` → kitni rows aur columns hain
- `.info()` → column names aur data types
- `.describe()` → har numeric column ka average, min, max, etc. (quick summary, kya range hai values ki)

In [3]:
print("Shape:", dataset.shape)
dataset.info()

Shape: (800, 7)
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   hours_studied          800 non-null    float64
 1   attendance_pct         800 non-null    float64
 2   sleep_hours            800 non-null    float64
 3   previous_score         800 non-null    float64
 4   practice_tests         800 non-null    int64  
 5   extracurricular_hours  800 non-null    float64
 6   final_score            800 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 43.9 KB


In [4]:
dataset.describe()

,hours_studied,attendance_pct,sleep_hours,previous_score,practice_tests,extracurricular_hours,final_score
count,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000
mean,5.974950,69.928487,6.538638,62.845338,7.053750,4.690237,79.668988
std,3.419152,17.541383,1.446717,18.607428,4.344474,2.840062,13.866826
min,0.040000,40.040000,4.020000,30.130000,0.000000,0.020000,38.270000
25%,2.987500,55.415000,5.300000,46.900000,3.000000,2.337500,69.575000
50%,6.065000,69.505000,6.510000,62.620000,7.000000,4.450000,80.340000
75%,9.002500,84.795000,7.810000,78.852500,11.000000,7.002500,91.050000
max,11.990000,99.970000,8.990000,94.960000,14.000000,9.990000,100.000000


## Step 4 — Input (X) aur Output (y) alag karo

- `X` = saare input features (jo model ko dikhayenge)
- `y` = `final_score` (jo predict karna hai)

In [5]:
X = dataset.drop(columns=["final_score"])
y = dataset["final_score"]

print("X columns:", list(X.columns))
print("y (target):", y.name)

X columns: ['hours_studied', 'attendance_pct', 'sleep_hours', 'previous_score', 'practice_tests', 'extracurricular_hours']
y (target): final_score


## Step 5 — Train/Test split

Data ko do parts mein baant rahe hain:
- **80% training data** → model isse seekhega
- **20% testing data** → model isse kabhi training ke time nahi dekhega, isi pe final check karenge ki model kitna accha predict karta hai

`random_state=42` fix rakha hai taaki har baar same split mile (reproducible result).

In [6]:
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", train_X.shape, " Test size:", test_X.shape)

Train size: (640, 6)  Test size: (160, 6)


## Step 6 — Feature Scaling

`StandardScaler` har column ko **same range** mein le aata hai (mean=0, std=1). Ye zaroori hai kyunki ANN training tab better/faster hoti hai jab saare inputs similar scale mein hon (warna badi values wale columns model ko "dominate" kar dete hain).

**Important:** Scaler ko `train_X` pe `fit_transform` karte hain, aur `test_X` pe sirf `transform` — taaki test data ka koi info training ke time "leak" na ho.

In [7]:
sc = StandardScaler()
train_X = sc.fit_transform(train_X)
test_X = sc.transform(test_X)

## Step 7 — Ek function banao jo ANN build + train + test kare

Taaki har activation ke liye baar-baar same code na likhna pade, ek function bana rahe hain: `build_and_test(activation_name)`.

**Model architecture (fixed rahega, sirf activation badlega):**
- Input layer: 6 units (6 features hain)
- Hidden layer 1: 6 units
- Hidden layer 2: 8 units
- Hidden layer 3: 4 units
- Output layer: 1 unit, **activation hamesha `linear`** (regression ke output layer mein linear hi hota hai, chahe hidden layers mein kuch bhi ho — kyunki final score jaisi koi bhi real-number value output karni hai)

Sirf **hidden layers ka activation** change hoga har baar (`relu`, `sigmoid`, `tanh`, `linear`).

In [8]:
def build_and_test(hidden_activation, epochs=60, batch_size=16):
    model = Sequential()

    model.add(Dense(units=6, kernel_initializer='glorot_uniform', input_dim=6))
    model.add(Activation(hidden_activation))

    model.add(Dense(units=8, kernel_initializer='glorot_uniform'))
    model.add(Activation(hidden_activation))

    model.add(Dense(units=4, kernel_initializer='glorot_uniform'))
    model.add(Activation(hidden_activation))

    model.add(Dense(units=1, kernel_initializer='glorot_uniform'))
    model.add(Activation('linear'))   # output layer -> regression ke liye hamesha linear

    model.compile(loss='mean_squared_error', optimizer='adam')
    model.fit(train_X, train_y, epochs=epochs, batch_size=batch_size, verbose=0)

    preds = model.predict(test_X, verbose=0).flatten()
    mse = mean_squared_error(test_y, preds)
    r2 = r2_score(test_y, preds)

    return model, preds, mse, r2

## Step 8 — Ek-ek karke har activation test karo

Ab har activation function ko **alag-alag** try karenge. Har cell ke baad dekhna:
- **Predictions vs Actual** — agar predictions sab ek jaisi/flat (constant) dikhein aur Actual se bahut door hon → us activation ne kuch seeka nahi (fail case).
- **R2 Score** — 1 ke jitna paas utna best. Negative ya 0 ke aas-paas matlab bahut kharab.

### Activation: `relu`

In [9]:
model_relu, preds_relu, mse_relu, r2_relu = build_and_test('relu')

print("Predictions (first 8):", preds_relu[:8].round(2))
print("Actual      (first 8):", test_y.values[:8])
print("MSE:", round(mse_relu, 3))
print("R2 Score:", round(r2_relu, 4))

f:\Deep_Learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Predictions (first 8): [ 89.34  96.02  61.26  85.89  69.89  67.37 115.11  68.72]
Actual      (first 8): [ 90.2  100.    78.99  83.95  70.94  63.77 100.    66.7 ]
MSE: 45.601
R2 Score: 0.7797


### Activation: `sigmoid`

In [10]:
model_sigmoid, preds_sigmoid, mse_sigmoid, r2_sigmoid = build_and_test('sigmoid')

print("Predictions (first 8):", preds_sigmoid[:8].round(2))
print("Actual      (first 8):", test_y.values[:8])
print("MSE:", round(mse_sigmoid, 3))
print("R2 Score:", round(r2_sigmoid, 4))

f:\Deep_Learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Predictions (first 8): [13.53 13.53 13.53 13.53 13.53 13.53 13.53 13.53]
Actual      (first 8): [ 90.2  100.    78.99  83.95  70.94  63.77 100.    66.7 ]
MSE: 4471.454
R2 Score: -20.5993


### Activation: `tanh`

In [11]:
model_tanh, preds_tanh, mse_tanh, r2_tanh = build_and_test('tanh')

print("Predictions (first 8):", preds_tanh[:8].round(2))
print("Actual      (first 8):", test_y.values[:8])
print("MSE:", round(mse_tanh, 3))
print("R2 Score:", round(r2_tanh, 4))

f:\Deep_Learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Predictions (first 8): [14.57 14.57 14.57 14.57 14.57 14.56 14.56 14.57]
Actual      (first 8): [ 90.2  100.    78.99  83.95  70.94  63.77 100.    66.7 ]
MSE: 4337.474
R2 Score: -19.9521


### Activation: `linear`

In [12]:
model_linear, preds_linear, mse_linear, r2_linear = build_and_test('linear')

print("Predictions (first 8):", preds_linear[:8].round(2))
print("Actual      (first 8):", test_y.values[:8])
print("MSE:", round(mse_linear, 3))
print("R2 Score:", round(r2_linear, 4))

f:\Deep_Learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Predictions (first 8): [ 85.43  94.33  68.73  87.75  69.69  68.37 106.86  73.37]
Actual      (first 8): [ 90.2  100.    78.99  83.95  70.94  63.77 100.    66.7 ]
MSE: 25.372
R2 Score: 0.8774


## Step 10 — Conclusion

- Jis activation ka **R2 sabse zyada (1 ke paas)** aur **MSE sabse kam** hai, wahi is dataset ke liye **best hidden-layer activation** hai — usi ko final model mein use karna hai.
- Generally `sigmoid` regression ke hidden layers mein **sabse kharab** perform karta hai (output ko 0-1 tak squeeze karke gradient weak kar deta hai, isliye predictions kaafi had tak flat/constant dikh sakte hain).
- `relu` zyadatar cases mein **best/most reliable** hota hai deep networks ke liye.
- `linear` bhi theek chal sakta hai agar data ka relation mostly linear ho, par complex/non-linear patterns capture nahi kar pata utna accha jitna relu karta hai.